# Silver Transactions — Verification Suite (`tests/02_silver_transactions_tests.ipynb`)

This notebook runs the 24-row synthetic fixture tests, rescued-data checks, and detailed Silver/quarantine read-back and rerun verification (`exceptAll == 0` in both directions) for `pipelines/02_silver_transactions.ipynb`.

In [ ]:
# Step 1: Configuration and Silver transformation logic matching pipelines/02_silver_transactions.ipynb
from datetime import date, datetime
from decimal import Decimal
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

BRONZE_TABLE = "workspace.sentinel_bronze.transactions"
SILVER_SCHEMA = "workspace.sentinel_silver"
SILVER_TRANSACTIONS_TABLE = f"{SILVER_SCHEMA}.transactions"
SILVER_QUARANTINE_TABLE = f"{SILVER_SCHEMA}.quarantine"

LINEAGE_COLUMNS = {
    "source_file",
    "source_row_index",
    "replay_batch",
    "_rescued_data",
    "_ingested_file_path",
    "_ingestion_timestamp",
}

EXCLUDED_PERSONAL_COLUMNS = {
    "first", "last", "gender", "street", "city", "state",
    "zip", "lat", "long", "city_pop", "job", "dob",
}

ACCEPTED_SILVER_COLUMNS = [
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

QUARANTINE_SILVER_COLUMNS = [
    "record_disposition", "duplicate_status", "duplicate_rank", "validation_reasons",
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "raw_trans_num", "raw_cc_num", "raw_merchant", "raw_amt",
    "raw_trans_date_trans_time", "unix_time", "_rescued_data",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]


def summarize_rescued_data(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Return rescued-data row counts and distinct JSON key combinations."""
    rescued_counts = df.agg(
        F.count("*").alias("input_rows"),
        F.count_if(F.col("_rescued_data").isNotNull()).alias("rows_with_non_null_rescued_data"),
        F.count_if(
            F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != "")
        ).alias("rows_with_non_blank_rescued_data"),
    )
    rescued_keys = (
        df.filter(F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""))
        .select(
            F.array_sort(
                F.map_keys(F.from_json(F.col("_rescued_data"), "MAP<STRING,STRING>"))
            ).alias("rescued_keys")
        )
        .groupBy("rescued_keys")
        .agg(F.count("*").alias("row_count"))
        .orderBy(F.col("row_count").desc())
    )
    return rescued_counts, rescued_keys


def classify_silver_transactions(bronze_df: DataFrame) -> DataFrame:
    """Execute the Silver type-conversion, field-validation, and duplicate-classification steps."""
    business_columns = sorted([
        col_name for col_name in bronze_df.columns if col_name not in LINEAGE_COLUMNS
    ])
    business_signature_struct = F.struct(*[
        F.coalesce(
            F.expr(f"NULLIF(TRIM(`{col_name}`), '')").cast("string"),
            F.lit("__NULL__"),
        ).alias(col_name)
        for col_name in business_columns
    ])

    typed_df = (
        bronze_df
        .withColumn("raw_trans_num", F.col("trans_num"))
        .withColumn("raw_cc_num", F.col("cc_num"))
        .withColumn("raw_merchant", F.col("merchant"))
        .withColumn("raw_amt", F.col("amt"))
        .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
        .withColumn("trans_num", F.expr("NULLIF(TRIM(trans_num), '')"))
        .withColumn("cc_num", F.expr("NULLIF(TRIM(cc_num), '')"))
        .withColumn("merchant", F.expr("NULLIF(TRIM(merchant), '')"))
        .withColumn("category", F.expr("NULLIF(TRIM(category), '')"))
        .withColumn("_normalized_event_time_str", F.expr("NULLIF(TRIM(trans_date_trans_time), '')"))
        .withColumn("event_time", F.expr("TRY_CAST(_normalized_event_time_str AS TIMESTAMP_NTZ)"))
        .withColumn("_normalized_amt_str", F.expr("NULLIF(TRIM(amt), '')"))
        .withColumn("_wide_amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(38, 18))"))
        .withColumn("amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(18, 2))"))
        .withColumn("_business_signature", F.to_json(business_signature_struct))
    )

    raw_validation_reasons = F.array(
        F.when(F.col("trans_num").isNull(), F.lit("missing_transaction_id")),
        F.when(F.col("cc_num").isNull(), F.lit("missing_card_id")),
        F.when(F.col("merchant").isNull(), F.lit("missing_merchant")),
        F.when(F.col("_normalized_event_time_str").isNull(), F.lit("missing_event_time")),
        F.when(
            F.col("_normalized_event_time_str").isNotNull() & F.col("event_time").isNull(),
            F.lit("unparseable_event_time"),
        ),
        F.when(F.col("_normalized_amt_str").isNull(), F.lit("missing_amount")),
        F.when(
            F.col("_normalized_amt_str").isNotNull()
            & (F.col("_wide_amount").isNull() | F.col("amount").isNull()),
            F.lit("unparseable_or_out_of_range_amount"),
        ),
        F.when(
            F.col("_wide_amount").isNotNull()
            & F.col("amount").isNotNull()
            & (F.col("_wide_amount") != F.col("amount").cast("decimal(38,18)")),
            F.lit("amount_precision_changed"),
        ),
        F.when(
            F.col("_wide_amount").isNotNull()
            & (F.col("_wide_amount") <= F.lit(0).cast("decimal(38,18)")),
            F.lit("non_positive_amount"),
        ),
        F.when(
            F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
            F.lit("unexpected_rescued_data"),
        ),
    )

    validated_df = typed_df.withColumn(
        "field_validation_reasons",
        F.filter(raw_validation_reasons, lambda reason: reason.isNotNull()),
    )

    duplicate_summary = (
        validated_df.filter(F.col("trans_num").isNotNull())
        .groupBy("trans_num")
        .agg(
            F.count("*").alias("id_occurrence_count"),
            F.countDistinct("_business_signature").alias("distinct_business_signatures"),
        )
    )

    id_order_window = Window.partitionBy("trans_num").orderBy(
        F.col("_ingestion_timestamp").asc_nulls_last(),
        F.col("_ingested_file_path").asc_nulls_last(),
        F.expr("TRY_CAST(source_row_index AS BIGINT)").asc_nulls_last(),
        F.col("source_row_index").asc_nulls_last(),
    )

    return (
        validated_df.join(duplicate_summary, on="trans_num", how="left")
        .withColumn(
            "duplicate_rank",
            F.when(F.col("trans_num").isNotNull(), F.row_number().over(id_order_window)).cast("int"),
        )
        .withColumn(
            "duplicate_status",
            F.when(F.col("trans_num").isNull(), F.lit("missing_id"))
            .when(F.col("id_occurrence_count") == 1, F.lit("unique_id"))
            .when(
                (F.col("id_occurrence_count") > 1) & (F.col("distinct_business_signatures") > 1),
                F.lit("conflicting_duplicate"),
            )
            .when(
                (F.col("id_occurrence_count") > 1)
                & (F.col("distinct_business_signatures") == 1)
                & (F.col("duplicate_rank") == 1),
                F.lit("identical_duplicate_primary"),
            )
            .otherwise(F.lit("identical_duplicate_extra")),
        )
        .withColumn(
            "duplicate_validation_reasons",
            F.when(
                F.col("duplicate_status") == "conflicting_duplicate",
                F.array(F.lit("conflicting_duplicate_transaction_id")),
            )
            .when(
                F.col("duplicate_status") == "identical_duplicate_extra",
                F.array(F.lit("duplicate_identical_transaction_id")),
            )
            .otherwise(F.array().cast("array<string>")),
        )
        .withColumn(
            "validation_reasons",
            F.concat(F.col("field_validation_reasons"), F.col("duplicate_validation_reasons")),
        )
        .withColumn(
            "record_disposition",
            F.when(F.size(F.col("validation_reasons")) == 0, F.lit("accepted"))
            .when(
                (F.col("duplicate_status") == "identical_duplicate_extra")
                & (F.size(F.col("field_validation_reasons")) == 0),
                F.lit("duplicate_identical"),
            )
            .otherwise(F.lit("quarantined")),
        )
    )


def assert_and_summarize_reconciliation(
    input_df: DataFrame, classified_df: DataFrame
) -> dict[str, int]:
    """Verify row-accounting and accepted-key invariants and return summary counts."""
    metrics_row = classified_df.agg(
        F.count("*").alias("classified_rows"),
        F.count_if(F.col("record_disposition").isNull()).alias("unclassified_rows"),
        F.count_if(F.col("record_disposition") == "accepted").alias("accepted_rows"),
        F.count_if(F.col("record_disposition") == "quarantined").alias("quarantined_rows"),
        F.count_if(F.col("record_disposition") == "duplicate_identical").alias("duplicate_identical_rows"),
        F.count_if(
            (F.col("record_disposition") == "accepted")
            & (F.col("trans_num").isNull() | (F.size(F.col("validation_reasons")) > 0))
        ).alias("invalid_accepted_rows"),
        F.count_if(
            (F.col("record_disposition") != "accepted") & (F.size(F.col("validation_reasons")) == 0)
        ).alias("non_accepted_without_reasons"),
        F.count_if(
            F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != "")
        ).alias("rows_with_rescued_data"),
    ).collect()[0]

    input_rows = input_df.count()
    distinct_accepted_ids = (
        classified_df.filter(F.col("record_disposition") == "accepted")
        .select("trans_num")
        .distinct()
        .count()
    )

    assert metrics_row["classified_rows"] == input_rows
    assert metrics_row["unclassified_rows"] == 0
    assert (
        metrics_row["accepted_rows"]
        + metrics_row["quarantined_rows"]
        + metrics_row["duplicate_identical_rows"]
        == input_rows
    )
    assert metrics_row["invalid_accepted_rows"] == 0
    assert metrics_row["non_accepted_without_reasons"] == 0
    assert distinct_accepted_ids == metrics_row["accepted_rows"]

    return {
        "input_rows": input_rows,
        "classified_rows": metrics_row["classified_rows"],
        "accepted_rows": metrics_row["accepted_rows"],
        "quarantined_rows": metrics_row["quarantined_rows"],
        "duplicate_identical_rows": metrics_row["duplicate_identical_rows"],
        "distinct_accepted_ids": distinct_accepted_ids,
        "rows_with_rescued_data": metrics_row["rows_with_rescued_data"],
    }

### Step 2: 24-Row Synthetic Fixture Verification

Exercises `classify_silver_transactions`, `summarize_rescued_data`, and `assert_and_summarize_reconciliation` on a controlled 24-row in-memory fixture covering valid records, whitespace normalization, missing/blank identifiers, missing/invalid timestamps, missing/malformed/non-positive/precision-changing amounts, identical and conflicting duplicates, multiple missing IDs, and rescued-data payloads.

In [ ]:
# Step 2: Build and verify the 24-row synthetic Bronze fixture in memory
synthetic_bronze_schema = T.StructType([
    T.StructField("trans_num", T.StringType(), True),
    T.StructField("cc_num", T.StringType(), True),
    T.StructField("merchant", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("amt", T.StringType(), True),
    T.StructField("trans_date_trans_time", T.StringType(), True),
    T.StructField("unix_time", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("source_row_index", T.StringType(), True),
    T.StructField("replay_batch", T.DateType(), True),
    T.StructField("_rescued_data", T.StringType(), True),
    T.StructField("_ingested_file_path", T.StringType(), True),
    T.StructField("_ingestion_timestamp", T.TimestampType(), True),
])


def make_synthetic_row(source_row_index: str, **overrides: object) -> dict:
    base_row = {
        "trans_num": f"TX_{source_row_index}",
        "cc_num": "4000000000000001",
        "merchant": "fraud_Kirlin and Sons",
        "category": "shopping_net",
        "amt": "42.50",
        "trans_date_trans_time": "2019-01-01 00:01:15",
        "unix_time": "1325376075",
        "source_file": "synthetic_fixture.csv",
        "source_row_index": source_row_index,
        "replay_batch": date(2019, 1, 1),
        "_rescued_data": None,
        "_ingested_file_path": "dbfs:/synthetic/part-00000.parquet",
        "_ingestion_timestamp": datetime(2026, 11, 5, 12, 0, 0),
    }
    base_row.update(overrides)
    return base_row


synthetic_rows = [
    make_synthetic_row("1", trans_num="TX_VALID_1", amt="42.50"),
    make_synthetic_row(
        "2",
        trans_num="  TX_VALID_TRIM  ",
        cc_num=" 4000000000000002 ",
        merchant=" fraud_Sporer-Keebler ",
        amt=" 125.00 ",
        trans_date_trans_time=" 2019-01-01 01:02:03 ",
    ),
    make_synthetic_row("3", trans_num="TX_NULL_BLANK_REQ", cc_num=None, merchant="   "),
    make_synthetic_row("4", trans_num="TX_TIME_MISSING", trans_date_trans_time="   "),
    make_synthetic_row("5", trans_num="TX_TIME_INVALID", trans_date_trans_time="2019-13-45 not-a-timestamp"),
    make_synthetic_row("6", trans_num="TX_AMT_MISSING", amt=""),
    make_synthetic_row("7", trans_num="TX_AMT_MALFORMED", amt="not_a_number"),
    make_synthetic_row("8", trans_num="TX_AMT_ZERO", amt="0.00"),
    make_synthetic_row("9", trans_num="TX_AMT_NEGATIVE", amt="-15.25"),
    make_synthetic_row("10", trans_num="TX_AMT_PRECISION", amt="19.999"),
    make_synthetic_row("11", trans_num="TX_AMT_NEG_PRECISION", amt="-0.004"),
    make_synthetic_row("12", trans_num="TX_DUP_IDENTICAL", amt="55.00", _ingestion_timestamp=datetime(2026, 11, 5, 12, 0, 0)),
    make_synthetic_row("13", trans_num=" TX_DUP_IDENTICAL ", amt="55.00", _ingestion_timestamp=datetime(2026, 11, 5, 12, 5, 0)),
    make_synthetic_row("14", trans_num="TX_DUP_IDENTICAL_INVALID", amt="-5.00"),
    make_synthetic_row("15", trans_num="TX_DUP_IDENTICAL_INVALID", amt="-5.00"),
    make_synthetic_row("16", trans_num="TX_DUP_CONFLICT_VALID", amt="25.00"),
    make_synthetic_row("17", trans_num="TX_DUP_CONFLICT_VALID", amt="99.00"),
    make_synthetic_row("18", trans_num="TX_DUP_CONFLICT_MIXED", amt="40.00"),
    make_synthetic_row("19", trans_num="TX_DUP_CONFLICT_MIXED", amt="corrupted"),
    make_synthetic_row("20", trans_num=None, amt="10.00"),
    make_synthetic_row("21", trans_num="   ", amt="20.00"),
    make_synthetic_row("22", trans_num="TX_RESCUED_JSON", _rescued_data='{"_unexpected_col":"extra_value"}'),
    make_synthetic_row("23", trans_num="TX_RESCUED_MALFORMED", _rescued_data="{malformed_json"),
    make_synthetic_row("24", trans_num="TX_RESCUED_WHITESPACE", _rescued_data="   "),
]

synthetic_bronze = spark.createDataFrame(synthetic_rows, schema=synthetic_bronze_schema)
synthetic_classified = classify_silver_transactions(synthetic_bronze)
synthetic_reconciliation = assert_and_summarize_reconciliation(synthetic_bronze, synthetic_classified)

assert synthetic_reconciliation == {
    "input_rows": 24,
    "classified_rows": 24,
    "accepted_rows": 4,
    "quarantined_rows": 19,
    "duplicate_identical_rows": 1,
    "distinct_accepted_ids": 4,
    "rows_with_rescued_data": 2,
}

results_by_index = {row["source_row_index"]: row.asDict() for row in synthetic_classified.collect()}

assert results_by_index["1"]["record_disposition"] == "accepted"
assert results_by_index["2"]["record_disposition"] == "accepted" and results_by_index["2"]["trans_num"] == "TX_VALID_TRIM"
assert results_by_index["3"]["validation_reasons"] == ["missing_card_id", "missing_merchant"]
assert results_by_index["4"]["validation_reasons"] == ["missing_event_time"]
assert results_by_index["5"]["validation_reasons"] == ["unparseable_event_time"]
assert results_by_index["6"]["validation_reasons"] == ["missing_amount"]
assert results_by_index["7"]["validation_reasons"] == ["unparseable_or_out_of_range_amount"]
assert results_by_index["8"]["validation_reasons"] == ["non_positive_amount"]
assert results_by_index["9"]["validation_reasons"] == ["non_positive_amount"]
assert results_by_index["10"]["validation_reasons"] == ["amount_precision_changed"]
assert results_by_index["11"]["validation_reasons"] == ["amount_precision_changed", "non_positive_amount"]
assert results_by_index["12"]["record_disposition"] == "accepted"
assert results_by_index["13"]["record_disposition"] == "duplicate_identical"
assert results_by_index["14"]["record_disposition"] == "quarantined"
assert results_by_index["15"]["validation_reasons"] == ["non_positive_amount", "duplicate_identical_transaction_id"]
assert results_by_index["16"]["validation_reasons"] == ["conflicting_duplicate_transaction_id"]
assert results_by_index["17"]["validation_reasons"] == ["conflicting_duplicate_transaction_id"]
assert results_by_index["18"]["validation_reasons"] == ["conflicting_duplicate_transaction_id"]
assert results_by_index["19"]["validation_reasons"] == ["unparseable_or_out_of_range_amount", "conflicting_duplicate_transaction_id"]
assert results_by_index["20"]["validation_reasons"] == ["missing_transaction_id"]
assert results_by_index["21"]["validation_reasons"] == ["missing_transaction_id"]
assert results_by_index["22"]["validation_reasons"] == ["unexpected_rescued_data"]
assert results_by_index["23"]["validation_reasons"] == ["unexpected_rescued_data"]
assert results_by_index["24"]["record_disposition"] == "accepted"

synthetic_rescued_counts, _ = summarize_rescued_data(synthetic_bronze)
assert synthetic_rescued_counts.collect()[0].asDict() == {
    "input_rows": 24,
    "rows_with_non_null_rescued_data": 3,
    "rows_with_non_blank_rescued_data": 2,
}

print("All 24 synthetic fixture rows passed classification and reconciliation checks.")

### Step 3: Detailed Silver Read-Back, `exceptAll` Content Equality & Rerun Verification

Compares the persisted `workspace.sentinel_silver.transactions` and `workspace.sentinel_silver.quarantine` tables against the freshly evaluated Bronze snapshot (`exceptAll == 0` in both directions), confirms zero synthetic fixture rows exist in either table, and verifies rerun idempotency.

In [ ]:
# Step 3: Verify persisted Silver & quarantine tables against expected outputs via two-way exceptAll
real_bronze_df = spark.table(BRONZE_TABLE)
assert real_bronze_df.filter(F.col("source_file") == "synthetic_fixture.csv").count() == 0

real_classified_df = classify_silver_transactions(real_bronze_df)
real_reconciliation = assert_and_summarize_reconciliation(real_bronze_df, real_classified_df)

expected_accepted_df = (
    real_classified_df.filter(F.col("record_disposition") == "accepted")
    .select(*ACCEPTED_SILVER_COLUMNS)
)
expected_quarantine_df = (
    real_classified_df.filter(F.col("record_disposition") != "accepted")
    .select(*QUARANTINE_SILVER_COLUMNS)
)


def assert_dataframe_content_equal(actual_df: DataFrame, expected_df: DataFrame, label: str) -> None:
    actual_fields = [(f.name, f.dataType) for f in actual_df.schema.fields]
    expected_fields = [(f.name, f.dataType) for f in expected_df.schema.fields]
    assert actual_fields == expected_fields, f"Schema mismatch for {label}"
    assert actual_df.count() == expected_df.count(), f"Count mismatch for {label}"
    assert actual_df.exceptAll(expected_df).count() == 0, f"Extra rows in {label}"
    assert expected_df.exceptAll(actual_df).count() == 0, f"Missing rows in {label}"


saved_accepted_df = spark.table(SILVER_TRANSACTIONS_TABLE)
saved_quarantine_df = spark.table(SILVER_QUARANTINE_TABLE)

assert saved_accepted_df.filter(F.col("source_file") == "synthetic_fixture.csv").count() == 0
assert saved_quarantine_df.filter(F.col("source_file") == "synthetic_fixture.csv").count() == 0

assert_dataframe_content_equal(saved_accepted_df, expected_accepted_df, SILVER_TRANSACTIONS_TABLE)
assert_dataframe_content_equal(saved_quarantine_df, expected_quarantine_df, SILVER_QUARANTINE_TABLE)

print(
    f"Detailed Silver verification passed: {saved_accepted_df.count():,} accepted rows, "
    f"{saved_quarantine_df.count():,} quarantine rows (0 synthetic rows, two-way exceptAll == 0)."
)